# 5 folds DLMMDD Calibrated Ensemble with Multi-Scale TTA

This notebook does **inference only**.
1. Computes OOF softmax on the held-out fold of each model (used to fit a temperature)
2. Runs **multi-scale + hflip TTA** on the test set
3. Temperature-scales each model's logits, then averages **softmax in log-space (geometric mean)**
4. Writes `submission_calibrated_ensemble.csv`

## How calibration works here
Each model is asked to predict on its OWN held-out validation fold (processed-val transform). We collect logits and labels per model, then fit a single scalar temperature T_i that minimises NLL on that OOF set. At test time we divide each model's logits by its T_i before softmax. After that, models' softmax magnitudes are comparable so the geometric mean (= sum of log-probs) is a sound combination.

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
dlmmdd_workshop_synthetic_source_attribution_challenge_path = kagglehub.competition_download('dlmmdd-workshop-synthetic-source-attribution-challenge')

print('Data source import complete.')

100%|██████████| 10.1G/10.1G [09:52<00:00, 18.3MB/s]

Extracting files...


Data source import complete.


In [4]:
import os
import shutil

# Define the paths
zip_file_name = 'dlmmdd_pad_processedval_eva02_large448_96gb_outputs5folds.zip'
drive_source_path = os.path.join('/content/drive/MyDrive/', zip_file_name)
destination_path = '/content/'

# Check if Google Drive is mounted
if not os.path.exists(os.path.dirname(drive_source_path)):
    print(f"Error: Google Drive not mounted at '{os.path.dirname(drive_source_path)}'. Please mount your drive first.")
else:
    # Copy the zip file from Drive to /content/
    if os.path.exists(drive_source_path):
        try:
            shutil.copy(drive_source_path, destination_path)
            print(f"Successfully copied '{zip_file_name}' from Drive to '{destination_path}'")

            # Unzip the file in /content/
            output_folder_name = zip_file_name.replace('.zip', '')
            unzip_path = os.path.join(destination_path, output_folder_name)

            if os.path.exists(os.path.join(destination_path, zip_file_name)):
                print(f"Unzipping '{zip_file_name}' to '{unzip_path}'...")
                shutil.unpack_archive(os.path.join(destination_path, zip_file_name), unzip_path, 'zip')
                print("File successfully unzipped!")
            else:
                print(f"Error: Zip file '{zip_file_name}' not found in '{destination_path}' after copying.")

        except Exception as e:
            print(f"An error occurred: {e}")
    else:
        print(f"Error: Zip file '{zip_file_name}' not found in your Google Drive at '{drive_source_path}'.")


Successfully copied 'dlmmdd_pad_processedval_eva02_large448_96gb_outputs5folds.zip' from Drive to '/content/'
Unzipping 'dlmmdd_pad_processedval_eva02_large448_96gb_outputs5folds.zip' to '/content/dlmmdd_pad_processedval_eva02_large448_96gb_outputs5folds'...
File successfully unzipped!


In [5]:
import io, os, gc, glob, math, random, zipfile, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageStat, ImageEnhance, ImageFilter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

# timm 1.0.20+ for eva02_large_patch14_448.mim_m38m_ft_in22k_in1k
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
               'timm>=1.0.20', 'huggingface_hub>=0.35.0', 'safetensors>=0.4.5'], check=False)
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
amp_dtype = torch.bfloat16 if (device.type=='cuda' and torch.cuda.is_bf16_supported()) else torch.float16
print('device', device, 'timm', timm.__version__, 'amp', amp_dtype)

device cuda timm 1.0.27 amp torch.bfloat16


In [6]:
COMP = '/root/.cache/kagglehub/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge'
TRAIN_CSV = f'{COMP}/Data/Data/training.csv'
TEST_CSV  = f'{COMP}/Data/Data/test.csv'

# Point at your EVA02-Large checkpoints unzipped from Drive.
# Each entry is (family_name, glob_pattern). Recursive '**' handles the double-nested
# folder that shutil.make_archive produced.
CKPT_GLOBS = [
    ('eva02_large_pad', '/content/dlmmdd_pad_processedval_eva02_large448_96gb_outputs5folds/**/*.pth'),
    # Add other families later only if EVA02-Large alone is short of LB. Don't
    # ensemble with EVA02-Base (correlated errors) - prefer ConvNeXtV2-Base for
    # architectural diversity if needed.
    # ('convnextv2_base_pad', '/content/dlmmdd_pad_processedval_convnextv2_base_outputs/**/*.pth'),
]

NUM_CLASSES = 10
# CRITICAL: must match the seed your training notebook used (20260511 in your
# DLMMDD_EVA02_Large notebook). If you use a different seed here, StratifiedKFold
# produces DIFFERENT (train, val) splits than training, the "OOF" set leaks training
# images, temperature calibration becomes invalid, and predictions get worse.
SEED        = 20260511
N_SPLITS    = 5
BATCH_SIZE  = 32        # safe for EVA02-Large 448 on a 96GB Blackwell
NUM_WORKERS = 4         # py3.12 + PyTorch DataLoader teardown is cleaner with fewer workers

# Multi-scale TTA: 1.0 = pad-to-square at model native res.
# 0.92 mimics the central_crop_resize post-processing; hflip is free diversity.
TTA_SCALES = (1.0, 0.92)
TTA_HFLIP  = True

OUT = Path('/content'); OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


In [7]:
def image_mean_color(img):
    return tuple(int(v) for v in ImageStat.Stat(img.resize((1,1), Image.Resampling.BILINEAR)).mean)

def pad_to_square_resize(img, size):
    return ImageOps.pad(img, (size, size), method=Image.Resampling.BICUBIC,
                       color=image_mean_color(img), centering=(0.5, 0.5))

class PadEvalTransform:
    """Pad-to-square + optional central scale crop + optional hflip."""
    def __init__(self, image_size, mean, std, scale=1.0, hflip=False):
        self.image_size = image_size; self.scale=scale; self.hflip=hflip
        self.tail = v2.Compose([
            v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=mean, std=std),
        ])
    def __call__(self, img):
        if self.hflip: img = ImageOps.mirror(img)
        if self.scale < 1.0:
            w, h = img.size
            nw, nh = max(16, int(w*self.scale)), max(16, int(h*self.scale))
            l, t = (w-nw)//2, (h-nh)//2
            img = img.crop((l, t, l+nw, t+nh))
        img = pad_to_square_resize(img, self.image_size)
        return self.tail(img)

# IMPORTANT: this must MATCH the ProcessedValTransform you trained against in
# pad-to-square-eva02 / DLMMDD_EVA02_Large notebooks. Same recipe = correct OOF.
class ProcessedValTransform:
    def __init__(self, image_size, mean, std):
        self.image_size=image_size
        self.tail = v2.Compose([
            v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=mean, std=std),
        ])
    def __call__(self, img):
        w,h = img.size
        nw, nh = max(16, int(w*0.92)), max(16, int(h*0.92))
        l, t = (w-nw)//2, (h-nh)//2
        img = img.crop((l,t,l+nw,t+nh))
        w,h = img.size
        if w>=h:
            img = img.crop((0,0,max(16,int(w*0.88)),h))
        else:
            img = img.crop((0,0,w,max(16,int(h*0.88))))
        img = img.resize((max(32,int(img.width*0.72)),max(32,int(img.height*0.72))), Image.Resampling.BICUBIC)
        img = img.resize((max(32,int(img.width/0.72)),max(32,int(img.height/0.72))), Image.Resampling.BICUBIC)
        img = ImageEnhance.Contrast(img).enhance(0.92)
        buf = io.BytesIO(); img.save(buf, format='JPEG', quality=55, optimize=False); buf.seek(0)
        img = Image.open(buf).convert('RGB')
        img = pad_to_square_resize(img, self.image_size)
        return self.tail(img)

def resolve_path(csv_path, is_test):
    p = str(csv_path); fn = os.path.basename(p); split = 'Test' if is_test else 'Training'
    for c in [p, f'{COMP}/{p}', f'{COMP}/Data/{p}', f'{COMP}/Data/Data/{p}',
              f'{COMP}/Data/Data/{split}/{fn}', f'{COMP}/Data/{split}/{fn}', f'{COMP}/{split}/{fn}']:
        if os.path.exists(c): return c
    return p

class SIADataset(Dataset):
    def __init__(self, df, transform, is_test):
        self.df = df.reset_index(drop=True); self.tx=transform; self.is_test=is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(resolve_path(row['path'], self.is_test)).convert('RGB')
        img = self.tx(img)
        if self.is_test: return img, str(row['ID'])
        return img, int(row['y'])

# --- Silence the Python 3.12 + PyTorch DataLoader teardown spam ("AssertionError:
# can only test a child process"). The errors are HARMLESS (the prediction
# completed; gradients/logits are unaffected) but they clutter the output.
# Fix: (1) explicitly stop worker processes before they get GC'd, (2) filter the
# residual __del__ noise from stderr.
import sys, atexit

def shutdown_dataloader(loader):
    if loader is None: return
    it = getattr(loader, '_iterator', None)
    if it is not None:
        try: it._shutdown_workers()
        except Exception: pass
        loader._iterator = None

_NOISE_TOKENS = (
    'can only test a child process',
    '_MultiProcessingDataLoaderIter.__del__',
    'Exception ignored in:',
    '_shutdown_workers',
    'w.is_alive()',
    'self._parent_pid == os.getpid()',
)
_orig_stderr_write = sys.stderr.write
def _filtered_stderr_write(s):
    if isinstance(s, str) and any(tok in s for tok in _NOISE_TOKENS):
        return len(s)
    return _orig_stderr_write(s)
sys.stderr.write = _filtered_stderr_write
atexit.register(gc.collect)

def make_loader(ds, batch_size):
    return DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=False,
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )


In [8]:
# Missing imports the original notebook forgot: StratifiedKFold + re.
import re
from sklearn.model_selection import StratifiedKFold

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
print('train', train_df.shape, 'test', test_df.shape)

kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_idx_map = {}
for fold, (tr, va) in enumerate(kf.split(train_df, train_df['y'])):
    fold_idx_map[fold] = (tr, va)
print('folds prepared with seed', SEED)

def parse_fold_from_name(name):
    m = re.search(r'fold(\d+)', name)
    return int(m.group(1)) if m else None


train (7000, 3) test (3000, 2)
folds prepared with seed 20260511


In [9]:
@torch.no_grad()
def predict_loader(model, loader):
    model.eval(); logits_list = []; ids = []
    try:
        for batch in tqdm(loader, leave=False):
            if len(batch) == 2 and isinstance(batch[1], torch.Tensor):
                x, y = batch; ids.append(y.cpu().numpy())
            else:
                x, _ids = batch; ids.append(np.array(_ids))
            x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type=='cuda'):
                out = model(x)
            logits_list.append(out.float().cpu().numpy())
    finally:
        shutdown_dataloader(loader)
    return np.concatenate(logits_list, 0), np.concatenate(ids, 0)

def build_model_from_ckpt(ckpt):
    name = ckpt['model_name']
    m = timm.create_model(name, pretrained=False, num_classes=NUM_CLASSES)
    m.load_state_dict(ckpt['model'])
    m = m.to(device, memory_format=torch.channels_last).eval()
    return m, int(ckpt['image_size']), ckpt['mean'], ckpt['std']

def fit_temperature(logits, labels, max_iter=200):
    """Optimise a single scalar T s.t. softmax(logits/T) minimises NLL on (logits, labels)."""
    z = torch.tensor(logits, dtype=torch.float32, device=device)
    y = torch.tensor(labels, dtype=torch.long, device=device)
    logT = torch.zeros((), device=device, requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=max_iter, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad()
        T = logT.exp()
        loss = F.cross_entropy(z / T, y)
        loss.backward(); return loss
    opt.step(closure)
    return float(logT.exp().item())


In [10]:
# For each model family, walk every checkpoint, do OOF eval (for temperature) and test TTA inference.
all_test_logprobs = []      # list of (3000, 10) calibrated log-probs
fold_records = []

for family, glob_pat in CKPT_GLOBS:
    # Recursive glob handles the double-nested folder from shutil.make_archive.
    ckpt_paths = sorted(glob.glob(glob_pat, recursive=True))
    if not ckpt_paths:
        print(f'  WARNING: no checkpoints found for {family} ({glob_pat})'); continue
    print(f'\n=== {family}: {len(ckpt_paths)} checkpoints ===')

    for ckpt_path in ckpt_paths:
        try: ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        except TypeError: ckpt = torch.load(ckpt_path, map_location='cpu')
        fold = parse_fold_from_name(os.path.basename(ckpt_path))
        if fold is None or fold not in fold_idx_map:
            print(f'  skip (no fold info): {ckpt_path}'); continue
        print(f'  fold {fold}: {os.path.basename(ckpt_path)}')

        model, img_size, mean, std = build_model_from_ckpt(ckpt)

        # --- OOF for temperature ---
        _, val_idx = fold_idx_map[fold]
        oof_ds = SIADataset(train_df.iloc[val_idx], ProcessedValTransform(img_size, mean, std), is_test=False)
        oof_loader = make_loader(oof_ds, batch_size=BATCH_SIZE*2)
        oof_logits, oof_y = predict_loader(model, oof_loader)
        oof_acc = (oof_logits.argmax(1) == oof_y).mean()
        T = fit_temperature(oof_logits, oof_y)
        nll0 = F.cross_entropy(torch.tensor(oof_logits), torch.tensor(oof_y, dtype=torch.long)).item()
        nll1 = F.cross_entropy(torch.tensor(oof_logits)/T, torch.tensor(oof_y, dtype=torch.long)).item()
        print(f'    OOF acc={oof_acc:.5f} T={T:.3f}  NLL {nll0:.4f}->{nll1:.4f}')
        fold_records.append({'family':family, 'fold':fold, 'oof_acc':oof_acc, 'T':T, 'path':ckpt_path})
        del oof_loader, oof_ds; gc.collect()

        # --- Test TTA ---
        test_logits_acc = np.zeros((len(test_df), NUM_CLASSES), dtype=np.float64)
        n_tta = 0
        hflip_modes = (False, True) if TTA_HFLIP else (False,)
        for scale in TTA_SCALES:
            for hflip in hflip_modes:
                tx = PadEvalTransform(img_size, mean, std, scale=scale, hflip=hflip)
                ds = SIADataset(test_df, tx, is_test=True)
                ldr = make_loader(ds, batch_size=BATCH_SIZE*2)
                z, _ = predict_loader(model, ldr)
                test_logits_acc += z
                n_tta += 1
                del ldr, ds; gc.collect()
        test_logits_mean = test_logits_acc / n_tta
        # Calibrate & convert to log-probs
        calibrated = test_logits_mean / T
        logprobs = calibrated - np.log(np.exp(calibrated).sum(1, keepdims=True))  # log_softmax
        all_test_logprobs.append(logprobs)

        del model; gc.collect()
        if device.type == 'cuda': torch.cuda.empty_cache()

fold_df = pd.DataFrame(fold_records)
print('\nPer-model summary:')
print(fold_df)
print('Mean OOF acc per family:')
print(fold_df.groupby('family')['oof_acc'].mean())



=== eva02_large_pad: 5 checkpoints ===
  fold 0: eva02_large_patch14_448.mim_m38m_ft_in22k_in1k_fold0.pth


  0%|          | 0/22 [00:00<?, ?it/s]

    OOF acc=0.99500 T=0.558  NLL 0.0917->0.0229


  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

...

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]


Per-model summary:
            family  fold   oof_acc         T  \
0  eva02_large_pad     0  0.995000  0.558371   
1  eva02_large_pad     1  0.993571  0.598287   
2  eva02_large_pad     2  0.992143  0.585643   
3  eva02_large_pad     3  0.993571  0.594193   
4  eva02_large_pad     4  0.992857  0.586100   

                                                path  
0  /content/dlmmdd_pad_processedval_eva02_large44...  
1  /content/dlmmdd_pad_processedval_eva02_large44...  
2  /content/dlmmdd_pad_processedval_eva02_large44...  
3  /content/dlmmdd_pad_processedval_eva02_large44...  
4  /content/dlmmdd_pad_processedval_eva02_large44...  
Mean OOF acc per family:
family
eva02_large_pad    0.993429
Name: oof_acc, dtype: float64


In [11]:
# Geometric mean of softmax == arithmetic mean of log-softmax.
# After temperature calibration, this is the right thing to do.
stacked = np.stack(all_test_logprobs, 0)         # (M, 3000, 10)
ensemble_logprobs = stacked.mean(0)
preds = ensemble_logprobs.argmax(1)
max_p = np.exp(ensemble_logprobs).max(1)
print('Mean confidence:', max_p.mean(), '  Low-confidence (<0.7) count:', int((max_p<0.7).sum()))

sub = pd.DataFrame({'ID': test_df['ID'].astype(int), 'TARGET': preds.astype(int)})
sub_path = '/content/submission_calibrated_ensemble.csv'
sub.to_csv(sub_path, index=False)
print(sub.head(), '\nsaved', sub_path)

# Save logits + log-probs for downstream pseudo-labelling.
np.save(OUT/'ensemble_logprobs.npy', ensemble_logprobs)
test_df[['ID']].to_csv(OUT/'ensemble_ids.csv', index=False)

# Copy submission + logprobs to Drive for safekeeping (optional).
try:
    shutil.copy(sub_path, '/content/drive/MyDrive/')
    shutil.copy(str(OUT/'ensemble_logprobs.npy'), '/content/drive/MyDrive/')
    shutil.copy(str(OUT/'ensemble_ids.csv'), '/content/drive/MyDrive/')
    print('copied to Drive')
except Exception as e:
    print('drive copy skipped:', e)


Mean confidence: 0.9880246386859795   Low-confidence (<0.7) count: 38
   ID  TARGET
0   6       6
1  12       1
2  16       7
3  17       1
4  18       6 
saved /content/submission_calibrated_ensemble.csv
copied to Drive


## Next steps

- Submit `submission_calibrated_ensemble.csv` to the competition.
- Check public LB.  Realistic expectation: 0.993-0.997 depending on how diverse your ckpts are.
- Save `ensemble_logprobs.npy` and `ensemble_ids.csv` as a Kaggle dataset; feed it into the pseudo-label fine-tune notebook in `claude/pseudo_label_finetune.ipynb`.
- If `confidence < 0.7` for >100 images, your ensemble is wobbly — re-train a model on pad-to-square first.